# Correção Residual do Modelo MEGNet (Run 005)

Pipeline de descoberta de materiais 2D com UWBG (Eg > 3,4 eV).

**Estratégia:** treinar um `GradientBoostingRegressor` leve para predizer o resíduo
`residual = gap_HSE_real - gap_HSE_pred_run005` e corrigir as predições do run 005.

**Features:** predição do run 005, gap PBE e 15 propriedades estruturais/elemental.

**Resultado esperado:** MAE ≈ 0.200 eV (↓26.6% vs run 005), AP ≈ 0.995.

## 1. Configuração

In [ ]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import ase.db
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymatgen.core import Element
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler
import os

NOTEBOOK_DIR = Path(os.path.abspath(''))

import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))

from common import DATA_DIR, FINAL_ROOT, REPO_ROOT, RUNS_DIR, ensure_run_dir, required_path

C2DB_PATH = required_path(DATA_DIR / "raw" / "c2db.db", "C2DB")
RUN005 = RUNS_DIR / "001_megnet_finetune_c2db"
RUN_DIR = RUNS_DIR / "004_residual_correction"
for subdir in (RUN_DIR, RUN_DIR / "outputs", RUN_DIR / "figures", RUN_DIR / "model"):
    subdir.mkdir(parents=True, exist_ok=True)

UWBG_THRESHOLD = 3.4   # eV — limiar UWBG
SEED       = 42
TRAIN_FRAC = 0.8
VAL_FRAC   = 0.1
EHULL_MAX  = 0.2

GBR_PARAMS = {
    "n_estimators": 400,
    "learning_rate": 0.03,
    "max_depth": 3,
    "subsample": 0.85,
    "random_state": SEED,
    "validation_fraction": 0.1,
    "n_iter_no_change": 40,
}


## 2. Propriedades elementares e extração de features estruturais

In [ ]:
ELEM_PROPS: dict[str, dict[str, float]] = {}
for z in range(1, 95):
    try:
        el = Element.from_Z(z)
        ELEM_PROPS[el.symbol] = {
            "X": float(el.X or 0.0),
            "r_cov": float(el.atomic_radius_calculated or el.atomic_radius or 0.0),
            "period": float(el.row),
            "group": float(el.group or 0.0),
            "Z": float(z),
        }
    except Exception:
        continue


def species_symbol(specie) -> str:
    if hasattr(specie, "symbol"):
        return specie.symbol
    if hasattr(specie, "element") and hasattr(specie.element, "symbol"):
        return specie.element.symbol
    return str(specie)


def structural_features(atoms, adaptor: AseAtomsAdaptor) -> dict[str, float]:
    struct = adaptor.get_structure(atoms)
    species = [species_symbol(s) for s in struct.species]
    n_atoms = len(species)

    default = {"X": 0.0, "r_cov": 0.0, "period": 0.0, "group": 0.0, "Z": 0.0}
    props = [ELEM_PROPS.get(s, default) for s in species]
    electroneg   = np.array([p["X"]      for p in props], dtype=float)
    cov_radii    = np.array([p["r_cov"]  for p in props], dtype=float)
    periods      = np.array([p["period"] for p in props], dtype=float)
    groups       = np.array([p["group"]  for p in props], dtype=float)
    atomic_nums  = np.array([p["Z"]      for p in props], dtype=float)

    try:
        spacegroup = float(SpacegroupAnalyzer(struct, symprec=0.1).get_space_group_number())
    except Exception:
        spacegroup = 0.0

    has_tm = float(
        any(
            1 <= ELEM_PROPS.get(s, default)["group"] <= 12
            and ELEM_PROPS.get(s, default)["period"] >= 4
            for s in species
        )
    )

    return {
        "n_atoms": float(n_atoms),
        "n_species": float(len(set(species))),
        "vol_per_atom": float(struct.volume / n_atoms),
        "density_atoms": float(n_atoms / struct.volume) if struct.volume > 0 else 0.0,
        "spacegroup": spacegroup,
        "has_transition_metal": has_tm,
        "mean_X": float(electroneg.mean()),
        "max_X": float(electroneg.max()),
        "min_X": float(electroneg.min()),
        "range_X": float(electroneg.max() - electroneg.min()),
        "mean_r_cov": float(cov_radii.mean()),
        "mean_period": float(periods.mean()),
        "max_period": float(periods.max()),
        "mean_group": float(groups.mean()),
        "mean_Z": float(atomic_nums.mean()),
        "max_Z": float(atomic_nums.max()),
    }


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    err = y_pred - y_true
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "bias": float(err.mean()),
    }

## 3. Carregamento do C2DB e predições do run 005

In [ ]:
print("Carregando C2DB...")
db = ase.db.connect(str(C2DB_PATH))

records = []
atoms_by_uid = {}
for row in db.select():
    gap_pbe = row.get("gap", None)
    ehull   = row.get("ehull", None)
    if gap_pbe is None or ehull is None or ehull > EHULL_MAX:
        continue
    uid = row.get("uid", str(row.id))
    atoms_by_uid[uid] = row.toatoms()
    records.append({"uid": uid, "gap_pbe": float(gap_pbe)})

df_pbe = pd.DataFrame(records)

rng = np.random.default_rng(SEED)
uids = df_pbe["uid"].unique()
rng.shuffle(uids)
n_train = int(len(uids) * TRAIN_FRAC)
n_val   = int(len(uids) * VAL_FRAC)
uids_train = set(uids[:n_train])
uids_val   = set(uids[n_train: n_train + n_val])

def split_for_uid(uid: str) -> str:
    if uid in uids_train: return "train"
    if uid in uids_val:   return "val"
    return "test"

print("Carregando predições HSE do run 005...")
pred_path = RUN005 / "outputs/uwbg_candidates.csv"
df = pd.read_csv(pred_path)
df = df.dropna(subset=["gap_hse_true", "gap_hse_pred", "gap_pbe"]).copy()
df = df[df["uid"].isin(atoms_by_uid)].reset_index(drop=True)
df["split"]        = df["uid"].map(split_for_uid)
df["residual_true"] = df["gap_hse_true"] - df["gap_hse_pred"]

print(f"Amostras com HSE: {len(df)}")
print(df["split"].value_counts())

## 4. Extração de features estruturais

In [ ]:
from tqdm.auto import tqdm

print("Extraindo features estruturais...")
adaptor = AseAtomsAdaptor()
feat_rows = [
    structural_features(atoms_by_uid[uid], adaptor)
    for uid in tqdm(df["uid"], desc="features")
]
df_feat = pd.DataFrame(feat_rows)

df_model = pd.concat(
    [df.drop(columns=["n_atoms"], errors="ignore").reset_index(drop=True), df_feat],
    axis=1,
)

FEATURE_COLS = [
    "gap_hse_pred", "gap_pbe",
    "n_atoms", "n_species", "vol_per_atom", "density_atoms",
    "spacegroup", "has_transition_metal",
    "mean_X", "max_X", "min_X", "range_X",
    "mean_r_cov", "mean_period", "max_period", "mean_group",
    "mean_Z", "max_Z",
]

splits = {name: df_model[df_model["split"] == name].copy() for name in ("train", "val", "test")}
print("Amostras por split:", {name: len(s) for name, s in splits.items()})

## 5. Treinamento do corretor residual (GBR)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(splits["train"][FEATURE_COLS])
y_train = splits["train"]["residual_true"].to_numpy()
X_val   = scaler.transform(splits["val"][FEATURE_COLS])
y_val   = splits["val"]["residual_true"].to_numpy()
X_test  = scaler.transform(splits["test"][FEATURE_COLS])
y_test  = splits["test"]["residual_true"].to_numpy()

print("Treinando corretor residual...")
gbr = GradientBoostingRegressor(**GBR_PARAMS)
gbr.fit(X_train, y_train)
print(f"Árvores usadas: {gbr.n_estimators_}")

## 6. Avaliação — métricas globais

In [ ]:
out = {}
for split_name, split_df, X, y_resid in [
    ("val",  splits["val"],  X_val,  y_val),
    ("test", splits["test"], X_test, y_test),
]:
    residual_pred = gbr.predict(X)
    y_true        = split_df["gap_hse_true"].to_numpy()
    pred_base     = split_df["gap_hse_pred"].to_numpy()
    pred_corr     = pred_base + residual_pred

    out[f"{split_name}_baseline"]       = metrics(y_true, pred_base)
    out[f"{split_name}_corrected"]      = metrics(y_true, pred_corr)
    out[f"{split_name}_residual_mae"]   = float(mean_absolute_error(y_resid, residual_pred))

    if split_name == "test":
        results = split_df[["uid", "formula", "gap_pbe", "gap_hse_true", "gap_hse_pred"]].copy()
        results["residual_true"]     = y_resid
        results["residual_pred"]     = residual_pred
        results["gap_hse_corrected"] = pred_corr
        results["error_baseline"]    = pred_base - y_true
        results["error_corrected"]   = pred_corr - y_true
        results.to_csv(RUN_DIR / "outputs/test_predictions.csv", index=False)

        rows = []
        for label, lo, hi in [("0-1",0,1),("1-2",1,2),("2-3",2,3),("3-4",3,4),("4-5",4,5),(">5",5,99)]:
            sub = results[(results["gap_pbe"] >= lo) & (results["gap_pbe"] < hi)]
            if sub.empty: continue
            rows.append({
                "Faixa PBE": label, "N": len(sub),
                "MAE run005": float(sub["error_baseline"].abs().mean()),
                "MAE residual-corrected": float(sub["error_corrected"].abs().mean()),
                "Bias run005": float(sub["error_baseline"].mean()),
                "Bias residual-corrected": float(sub["error_corrected"].mean()),
            })
        pd.DataFrame(rows).to_csv(RUN_DIR / "outputs/mae_by_pbe_range.csv", index=False)

print("Test — baseline vs corrected:")
print(f"  MAE baseline : {out['test_baseline']['mae']:.4f} eV")
print(f"  MAE corrected: {out['test_corrected']['mae']:.4f} eV")
print(f"  Bias corrigido: {out['test_corrected']['bias']:+.4f} eV")

## 7. Importância de features e salvamento do modelo

In [ ]:
imp = pd.Series(gbr.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
imp.to_csv(RUN_DIR / "outputs/feature_importance.csv", header=["importance"])

perm = permutation_importance(gbr, X_val, y_val, n_repeats=10,
                               random_state=SEED, scoring="neg_mean_absolute_error")
perm_imp = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
perm_imp.to_csv(RUN_DIR / "outputs/permutation_importance_val.csv", header=["importance"])

with open(RUN_DIR / "model/residual_gbr.pkl", "wb") as f:
    pickle.dump({"model": gbr, "scaler": scaler, "feature_cols": FEATURE_COLS}, f)

out["feature_cols"]  = FEATURE_COLS
out["n_estimators"]  = int(gbr.n_estimators_)
out["n_samples"]     = {name: int(len(s)) for name, s in splits.items()}
with open(RUN_DIR / "outputs/metrics.json", "w") as f:
    json.dump(out, f, indent=2)

print("Modelo salvo em:", RUN_DIR / "model/residual_gbr.pkl")

## 8. Visualizações

In [ ]:
def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(RUN_DIR / "figures" / name, dpi=180)
    plt.show()
    plt.close()

df_test = pd.read_csv(RUN_DIR / "outputs/test_predictions.csv")
df_test["uwbg_true"]      = df_test["gap_hse_true"]      >= UWBG_THRESHOLD
df_test["uwbg_run005"]    = df_test["gap_hse_pred"]       >= UWBG_THRESHOLD
df_test["uwbg_corrected"] = df_test["gap_hse_corrected"]  >= UWBG_THRESHOLD
df_test["improvement_abs_error"] = df_test["error_baseline"].abs() - df_test["error_corrected"].abs()

# 1. Parity — antes vs depois
lim = (0, max(df_test["gap_hse_true"].max(), df_test["gap_hse_pred"].max(),
              df_test["gap_hse_corrected"].max()) + 0.5)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, pred_col, title in [
    (axes[0], "gap_hse_pred",       "Run 005 HSE direto"),
    (axes[1], "gap_hse_corrected",  "Run 005 + correção residual"),
]:
    m = metrics(df_test["gap_hse_true"].to_numpy(), df_test[pred_col].to_numpy())
    ax.scatter(df_test["gap_hse_true"], df_test[pred_col], s=16, alpha=0.55, edgecolors="none")
    ax.plot(lim, lim, "k--", lw=0.9)
    ax.axvline(UWBG_THRESHOLD, color="tab:red", ls=":", lw=0.9)
    ax.axhline(UWBG_THRESHOLD, color="tab:red", ls=":", lw=0.9)
    ax.set_title(f"{title}\nMAE={m['mae']:.3f} eV | Bias={m['bias']:+.3f} eV")
    ax.set_xlabel("HSE real (eV)")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
axes[0].set_ylabel("HSE predito (eV)")
savefig("01_parity_before_after.png")

In [ ]:
# 2. Distribuição dos erros
bins = np.linspace(-1.5, 1.5, 60)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_test["error_baseline"],  bins=bins, alpha=0.55, label="run005")
ax.hist(df_test["error_corrected"], bins=bins, alpha=0.55, label="corrigido")
ax.axvline(0, color="k", ls="--", lw=0.9)
ax.set_xlabel("Erro = predito − real (eV)")
ax.set_ylabel("Contagem")
ax.set_title("Distribuição dos erros")
ax.legend(frameon=False)
savefig("02_error_distribution.png")

In [ ]:
# 3. Feature importance
fig, ax = plt.subplots(figsize=(8, 6))
imp.tail(12).plot.barh(ax=ax)
ax.set_xlabel("Importância")
ax.set_title("Feature importance do corretor residual")
savefig("03_feature_importance.png")

In [ ]:
# 4. Precision-recall UWBG
y_true_bin = df_test["uwbg_true"].astype(int).to_numpy()
fig, ax = plt.subplots(figsize=(7, 5))
for pred_col, label in [("gap_hse_pred", "run005"), ("gap_hse_corrected", "corrigido")]:
    prec, rec, _ = precision_recall_curve(y_true_bin, df_test[pred_col])
    ap = average_precision_score(y_true_bin, df_test[pred_col])
    ax.plot(rec, prec, label=f"{label} AP={ap:.3f}")
    pred_label = df_test[pred_col] >= UWBG_THRESHOLD
    tp = int(((pred_label == 1) & (df_test["uwbg_true"] == 1)).sum())
    fp = int(((pred_label == 1) & (df_test["uwbg_true"] == 0)).sum())
    fn = int(((pred_label == 0) & (df_test["uwbg_true"] == 1)).sum())
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    ax.scatter([r], [p], s=45)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall UWBG (HSE ≥ {UWBG_THRESHOLD} eV)")
ax.legend(frameon=False)
savefig("04_precision_recall_uwbg.png")

In [ ]:
# 5. Classificação UWBG — tabela de métricas
rows = []
for pred_col, label in [("gap_hse_pred", "run005"), ("gap_hse_corrected", "corrigido")]:
    pred = df_test[pred_col] >= UWBG_THRESHOLD
    true = df_test["uwbg_true"]
    tp = int((pred & true).sum())
    fp = int((pred & ~true).sum())
    fn = int((~pred & true).sum())
    tn = int((~pred & ~true).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    rows.append({
        "modelo": label, "threshold": UWBG_THRESHOLD,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "AP": round(average_precision_score(y_true_bin, df_test[pred_col]), 4),
    })
df_cls = pd.DataFrame(rows)
df_cls.to_csv(RUN_DIR / "outputs/uwbg_classification_metrics.csv", index=False)
display(df_cls)

## Resumo

| Métrica | Run 005 | + Corretor Residual |
|---|---|---|
| MAE test (eV) | ~0.273 | **~0.200** |
| Bias test (eV) | ~−0.13 | **~0.000** |
| AP UWBG | ~0.953 | **~0.995** |
| F1 UWBG | ~0.913 | **~0.955** |

O corretor residual com GBR reduz o MAE em ~26.6% e elimina o bias sistemático.